# 22DM015 Final Project — Financial PhraseBank
## Person B: Part 3 — Full-dataset SOA comparison

**Dataset:** `takala/financial_phrasebank`, config `sentences_allagree` (2,264 sentences).‍
**Labels:** 0 = negative, 1 = neutral, 2 = positive.‍

### Shared data contract (set by Person D — do NOT re-split)
- Splits are committed under `data/` as CSVs: `train` (1584), `val` (227), `test` (453); `labeled_32.csv` is loaded separately.‍
- Evaluate headline numbers on **`test`** only.‍ Use `eval_utils.evaluate` so we're comparable.‍
- Log every result with `eval_utils.log_result(...)` into `results/results.csv`.‍
- Part 2 (BERT fine-tuning + augmentation) lives in `notebooks/bert_part2.ipynb`; this notebook reads its rows from the shared scoreboard to overlay them on the curve.‍

### What this notebook does
- **3a.** Train on 1 / 10 / 25 / 50 / 75 / 100 % of `train` (use `du.subset_by_fraction`).‍
- **3b.** Plot the learning curve.‍
- **3c.** Fold in Part 2 techniques (read from `results/results.csv`).‍
- **3d.** Methodology analysis (student-authored).‍

> **AI disclosure:** any AI-generated code/text in this notebook must be watermarked and declared (course rule).‍ Interpretation, methodology justification, and analysis must be student-authored.‍

In [ ]:
# watermark: AGLLM (AI-assisted content disclosure)
# --- Reproducibility seed (required by the assignment) ---
import os, random, sys
import numpy as np
import pandas as pd
SEED = 618
random.seed(SEED); np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# Make the shared helpers importable (they live in the repo root, one level up).
sys.path.append(os.path.abspath('..'))
import data_utils as du
import eval_utils as eu

splits = du.load_splits()            # identical train/val/test for everyone
train, val, test = splits['train'], splits['val'], splits['test']
# Part 3 keeps val HELD OUT (not folded into train): every Part 3 training early-stops on
# val and keeps the best-on-val checkpoint, so val must stay separate. The full-data curve
# point is therefore 1584, and the headline numbers are still measured on test only.
for k, v in splits.items():
    print(f'{k:11s} {len(v):5d}', dict(v['label_name'].value_counts()))

> **Install (run once):** `transformers`, `torch`, `accelerate` are needed here.‍ On Python 3.14 torch wheels may be missing — use a 3.11/3.12 venv.‍

## Part 3 setup — shared training helpers
Same training protocol as `bert_part2.ipynb` so the curve points are directly comparable to the Part 2 rows that get overlaid in 3c.‍ The helpers (`train_bert`, `eval_split`, `logged`, `notes_kv`, `fmt`) are duplicated here on purpose: the two notebooks remain runnable independently.‍

In [ ]:
# watermark: AGLLM (AI-assisted content disclosure)
# Shared training helpers (mirrors bert_part2.ipynb so Part 3 stays runnable on its own).
import logging, warnings
# Tell transformers to skip its TensorFlow/Flax/Keras probe (we use PyTorch only).
os.environ.setdefault('USE_TF', '0')
os.environ.setdefault('USE_FLAX', '0')
os.environ.setdefault('TRANSFORMERS_NO_ADVISORY_WARNINGS', '1')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
logging.getLogger('tensorflow').setLevel(logging.ERROR)
logging.getLogger('torch.distributed.elastic.multiprocessing.redirects').setLevel(logging.ERROR)
logging.getLogger('torchao').setLevel(logging.ERROR)
warnings.filterwarnings('ignore', message=r'.*Skipping import of cpp extensions.*')

import torch
from datasets import Dataset
from sklearn.metrics import f1_score
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          Trainer, TrainingArguments, set_seed, EarlyStoppingCallback)

torch.set_num_threads(os.cpu_count() or 4)   # all trainings here are CPU-bound
MODEL = 'bert-base-uncased'      # same as Part 2 for direct comparability
NUM_LABELS = 3
MAX_LEN = 128

tok = AutoTokenizer.from_pretrained(MODEL)


def encode(df, max_len=MAX_LEN):
    ds = Dataset.from_pandas(df[['text', 'label']], preserve_index=False)
    return ds.map(lambda b: tok(b['text'], truncation=True, padding='max_length', max_length=max_len),
                  batched=True)


def compute_metrics(eval_pred):
    """macro-F1 on the eval (val) split — the quantity early stopping watches."""
    logits, labels = eval_pred
    return {'f1_macro': f1_score(labels, logits.argmax(-1), average='macro')}


def set_trainable(model, freeze_base):
    """Freeze (feature-extraction) or unfreeze (full fine-tuning) the transformer backbone;
    the classification head always stays trainable. model.base_model is the encoder for
    BERT/DistilBERT and the head sits outside it."""
    for p in model.base_model.parameters():
        p.requires_grad = not freeze_base


def train_bert(train_df, out_dir, *, epochs=6, batch=16, max_len=MAX_LEN, log_epochs=False,
               mode='full', warmup_frac=0.5, patience=2):
    """Shared fine-tuning protocol with EARLY STOPPING on the held-out val split: train up
    to `epochs`, evaluate macro-F1 on val each epoch, keep the best-on-val checkpoint, and
    stop after `patience` epochs without improvement. mode controls which weights update:
      'full'   - unfreeze everything (default; what the curve and Part 2 runs use),
      'frozen' - freeze BERT, train only the classifier head (feature extraction),
      'warmup' - head only with BERT frozen for warmup_frac of the epochs, then unfreeze."""
    set_seed(SEED)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=NUM_LABELS)
    train_ds = encode(train_df, max_len)
    val_ds = encode(val, max_len)

    def _run(n_epochs, freeze):
        if n_epochs <= 0:
            return None
        set_trainable(model, freeze)
        args = TrainingArguments(
            output_dir=out_dir, seed=SEED,
            num_train_epochs=n_epochs, per_device_train_batch_size=batch,
            per_device_eval_batch_size=64, learning_rate=2e-5,
            eval_strategy='epoch', save_strategy='epoch', save_total_limit=1,
            load_best_model_at_end=True, metric_for_best_model='f1_macro', greater_is_better=True,
            logging_strategy='epoch' if log_epochs else 'no',
            report_to='none', disable_tqdm=True,
        )
        tr = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                     compute_metrics=compute_metrics,
                     callbacks=[EarlyStoppingCallback(early_stopping_patience=patience)])
        tr.train()
        return tr

    if mode == 'frozen':
        trainer = _run(epochs, freeze=True)
    elif mode == 'warmup':
        warm = max(1, min(epochs - 1, round(epochs * warmup_frac)))
        _run(warm, freeze=True)                       # phase 1: classifier head only
        trainer = _run(epochs - warm, freeze=False)   # phase 2: full fine-tune
    else:                                             # 'full'
        trainer = _run(epochs, freeze=False)
    trainer.eval_max_len = max_len
    return trainer


def eval_split(trainer, df, max_len=None):
    max_len = max_len or getattr(trainer, 'eval_max_len', MAX_LEN)
    pred = trainer.predict(encode(df, max_len)).predictions.argmax(-1)
    return eu.evaluate(df['label'].values, pred)


def logged(method, full_row=False):
    """Latest TEST row for (MODEL, method) from the shared scoreboard. The eval module
    keys rows on (model, method, split, n_train_labeled) and no longer tracks person, so
    we match on MODEL + method here (each method has a single n in this notebook).
    Delete a row from results.csv to force that experiment to re-run."""
    if not eu.RESULTS_CSV.exists():
        return None
    r = pd.read_csv(eu.RESULTS_CSV)
    r = r[(r['model'] == MODEL) & (r['method'] == method) & (r['split'] == 'test')]
    if not len(r):
        return None
    row = r.iloc[-1]
    return row if full_row else {k: row[k] for k in eu.METRIC_KEYS}


def notes_kv(notes):
    """Parse the 'k=v; k=v' segments of a notes string into a dict."""
    out = {}
    for seg in str(notes).split(';'):
        if '=' in seg:
            k, v = seg.split('=', 1)
            out[k.strip()] = v.strip()
    return out


fmt = eu.fmt

## Part 3a.‍ Data-scaling curve (1 / 10 / 25 / 50 / 75 / 100 %)

In [ ]:
# watermark: AGLLM (AI-assisted content disclosure)
# Part 3a: data-scaling curve — fine-tune a fresh BERT on increasing fractions of train,
# early-stopping on val, and evaluate the best-on-val checkpoint on test. Lighter fixed
# protocol (max_len 64, batch 16, up to 6 epochs, early-stop patience 2) so the curve
# isolates data quantity.
# RESUME-AWARE + PROTOCOL-SAFE: a logged fraction is reused only if its notes match the
# current CURVE protocol (epochs/maxlen/patience) — editing CURVE retrains instead of
# silently mixing protocols. With early stopping this is ~2-3 h on CPU; delete rows to force.
# NOTE: du.subset_by_fraction samples each fraction independently (stratified, same
# seed) — fractions are NOT nested subsets of each other; see its docstring.
CURVE = dict(epochs=6, batch=16, max_len=64, patience=2)
FRACTIONS = [0.01, 0.10, 0.25, 0.50, 0.75, 1.00]

curve_rows = []
for f in FRACTIONS:
    method = f'full-{int(f * 100)}%'
    sub = du.subset_by_fraction(train, f)
    prev = logged(method, full_row=True)
    m = None
    if prev is not None:
        kv = notes_kv(prev['notes'])
        if (kv.get('epochs') == str(CURVE['epochs']) and kv.get('maxlen') == str(CURVE['max_len'])
                and kv.get('patience') == str(CURVE['patience'])):
            m = {k: prev[k] for k in eu.METRIC_KEYS}
            print(f"frac={f:>4}  n={len(sub):4d}  [cached]  macroF1={float(m['f1_macro']):.4f}")
        else:
            print(f"frac={f:>4}  [stale] logged row used a different protocol — retraining")
    if m is None:
        tr = train_bert(sub, '../.cache/bert_curve', **CURVE)
        m = eval_split(tr, test)
        eu.log_result(MODEL, method, len(sub), m,
                      notes=f"frac={f}; epochs={CURVE['epochs']}; maxlen={CURVE['max_len']}; "
                            f"patience={CURVE['patience']}")
        print(f"frac={f:>4}  n={len(sub):4d}  [trained] macroF1={float(m['f1_macro']):.4f}")
    curve_rows.append({'frac': f, 'n_train': len(sub), **m})

pd.DataFrame(curve_rows)

## Part 3c (apples-to-apples) — Part 2 techniques under the curve's training protocol
The 3c overlay below plots the Part 2 rows at their own protocol (`max_len 128`, batch 8, 20 epochs), which is not the curve's (`max_len 64`, batch 16, up to 6 epochs with early stopping on val), so the points are not strictly comparable.‍ To put every method on one schema, re-train each Part 2 technique with the curve protocol and log it under a `-curveproto` suffix.‍ `part3d.ipynb` reads these rows to compute an exact "real-label equivalent" for each technique.‍

In [ ]:
# watermark: AGLLM (AI-assisted content disclosure)
# Part 3c (apples-to-apples): re-train each Part 2 technique under the SAME protocol as the
# data-scaling curve (CURVE = max_len 64 / batch 16 / up to 6 epochs, early-stop on val,
# patience 2), so every point compared in part3d sits on one training schema. The Part 2
# deliverable rows (128/8/20) logged by bert_part2.ipynb are left untouched; these go in
# under a '-curveproto' method suffix and are used only for the Part 3 comparison.
# RESUME-AWARE + PROTOCOL-SAFE: a '-curveproto' row is reused only if its notes record the
# current CURVE epochs/maxlen/patience. Delete rows to force a re-train.
l32 = pd.read_csv('../data/labeled_32.csv')
aug = pd.read_csv('../data/augmented_32.csv')
gen = pd.read_csv('../data/llm_generated.csv')
combo_llm = pd.concat([l32[['text', 'label']], gen[['text', 'label']]], ignore_index=True)
combo_opt = (pd.concat([l32[['text', 'label']], aug[['text', 'label']], gen[['text', 'label']]],
                       ignore_index=True)
             .drop_duplicates(subset='text', ignore_index=True))

# (base method name, training frame) — mirrors the Part 2 recipes exactly.
PART2_SETS = [('32-shot', l32), ('augmented', aug),
              ('llm-generated', combo_llm), ('optimal-combo', combo_opt)]

cp_rows = []
for base, df in PART2_SETS:
    method = f'{base}-curveproto'
    prev = logged(method, full_row=True)
    kv = notes_kv(prev['notes']) if prev is not None else {}
    if (kv.get('epochs') == str(CURVE['epochs']) and kv.get('maxlen') == str(CURVE['max_len'])
            and kv.get('patience') == str(CURVE['patience'])):
        m = {k: prev[k] for k in eu.METRIC_KEYS}
        print(f"{base:14s} n={len(df):4d} [cached]  macroF1={float(m['f1_macro']):.4f}")
    else:
        tr = train_bert(df, '../.cache/bert_p2_curveproto', **CURVE)
        m = eval_split(tr, test)
        eu.log_result(MODEL, method, len(df), m,
                      notes=f"part2 technique on curve protocol; base={base}; "
                            f"epochs={CURVE['epochs']}; maxlen={CURVE['max_len']}; "
                            f"patience={CURVE['patience']}")
        print(f"{base:14s} n={len(df):4d} [trained] macroF1={float(m['f1_macro']):.4f}")
    cp_rows.append({'base': base, 'n': len(df), **m})

pd.DataFrame(cp_rows)

## Part 3 extra — freeze vs full fine-tune vs warm-up (the few-shot stability question)
Full fine-tuning updates all ~110M BERT parameters; on only 32 examples that is unstable.‍ The alternatives are to freeze BERT and train just the ~2,300-parameter classifier head (feature extraction), or to warm up the head with BERT frozen and then unfreeze everything.‍ This cell compares the three modes (via the `mode` argument of `train_bert`) on the 32 labelled examples at the curve protocol.‍

In [ ]:
# watermark: AGLLM (AI-assisted content disclosure)
# Freeze vs full fine-tune vs warm-up, on the 32 labelled examples — the regime where the
# choice matters most (full fine-tuning ~110M params on 32 rows is unstable). Same curve
# protocol (up to 6 epochs / maxlen 64, early-stop on val, patience 2). 'frozen' trains only
# the ~2.3k-parameter classifier head (feature extraction); 'warmup' trains the head with
# BERT frozen for half the epochs, then unfreezes. RESUME-AWARE. Needs l32 from the cell above.
freeze_rows = []
for label, mode in [('full fine-tune', 'full'), ('frozen head', 'frozen'),
                    ('warm-up then full', 'warmup')]:
    method = f'32shot-{mode}'
    prev = logged(method, full_row=True)
    if (prev is not None and notes_kv(prev['notes']).get('maxlen') == str(CURVE['max_len'])
            and notes_kv(prev['notes']).get('patience') == str(CURVE['patience'])):
        m = {k: prev[k] for k in eu.METRIC_KEYS}
        print(f'{label:18s} [cached]  macroF1={float(m["f1_macro"]):.4f}')
    else:
        tr = train_bert(l32, '../.cache/bert_freezedemo', mode=mode, **CURVE)
        m = eval_split(tr, test)
        eu.log_result(MODEL, method, len(l32), m,
                      notes=f"32-shot training-mode demo; mode={mode}; "
                            f"epochs={CURVE['epochs']}; maxlen={CURVE['max_len']}; "
                            f"patience={CURVE['patience']}")
        print(f'{label:18s} [trained] macroF1={float(m["f1_macro"]):.4f}')
    freeze_rows.append({'mode': label, **m})

pd.DataFrame(freeze_rows)[['mode', 'accuracy', 'f1_macro', 'f1_negative', 'f1_neutral', 'f1_positive']]

### What the freeze comparison shows
At the curve's minimal budget (32 examples, 3 epochs) none of the three modes learns the task well, but they differ: full fine-tuning reaches 0.237 macro-F1, warm-up 0.167 and the frozen head only 0.162.‍ Freezing does not help here, and the reason is the budget rather than the idea.‍ With only the ~2,300-parameter head training on 32 rows, three epochs is about six gradient steps — far too few for the head to converge — and the frozen `[CLS]` features of an un-fine-tuned BERT are not adapted to the task.‍ Feature extraction is cheap enough to run for many more epochs, and that is where it would become competitive and more stable; at three epochs full fine-tuning, despite updating all 110M parameters, simply moves further.‍ So the freeze / full / warm-up choice is a real lever (now exposed through the `mode` argument of `train_bert`), but on this dataset at this protocol full fine-tuning is not the unstable loser the few-shot intuition suggests — the instability shows up instead as uniformly low scores.‍ A fair test of freezing would give the frozen head many more (cheap) epochs.‍

## Part 3b / 3c.‍ Learning curve + Part 2 techniques overlay

In [ ]:
# watermark: AGLLM (AI-assisted content disclosure)
# Part 3b: learning curve from the shared results scoreboard.
# Part 3c: overlay every Part 2 technique logged in results.csv on the same axes, each at
# the number of training rows it used — i.e. "how many real labels is each technique worth?"
# Person C's LLM rows (zero-/few-shot) and Person A's Part 1 baselines are drawn as
# horizontal reference lines (no x-position on the log axis). Baselines fall back to the
# Part 1 notebook's numbers until Person A logs them through eu.log_result.
import matplotlib.pyplot as plt

res = pd.read_csv(eu.RESULTS_CSV)
bsel = (res['model'] == MODEL)
# log_result upserts on (model, method, split, n_train_labeled), so rows are already unique
curve = res[bsel & res['method'].str.startswith('full-')].sort_values('n_train_labeled')

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(curve['n_train_labeled'], curve['f1_macro'], marker='o', label='BERT macro-F1 (data-scaling)')
ax.plot(curve['n_train_labeled'], curve['accuracy'], marker='s', ls='--', label='BERT accuracy')

# Part 2 techniques (3c) — only the ones already logged are drawn, so this cell
# degrades gracefully until 2d/2e have run
TECHNIQUES = [('32-shot', 'red', '2a 32-shot'),
              ('augmented', 'darkorange', '2b augmented (back-translation)'),
              ('llm-generated', 'purple', '2d 32 + LLM-generated'),
              ('optimal-combo', 'saddlebrown', '2e optimal combo')]
tech_rows = []
for method, color, label in TECHNIQUES:
    row = res[bsel & (res['method'] == method)]
    if len(row):
        ax.scatter(row['n_train_labeled'], row['f1_macro'], color=color, zorder=5,
                   label=f'{label} macro-F1')
        tech_rows.append(row)

# Person C's zero-/few-shot LLM rows (Part 2c) as horizontal reference lines
llm = res[res['method'].isin(['zero-shot', 'few-shot'])]
for _, r in llm.iterrows():
    ax.axhline(float(r['f1_macro']), ls='-.', lw=1,
               label=f"{r['model']} {r['method']} (macro-F1 {float(r['f1_macro']):.2f})")

# Part 1 reference floors (test split): read Person A's logged rows when available,
# otherwise fall back to the numbers from the Part 1 notebook
for pattern, fallback, color, label in [('random', 0.3035, 'gray', 'random floor'),
                                        ('rule', 0.7304, 'green', 'rule-based')]:
    row = res[res['method'].str.contains(pattern, case=False, na=False)]
    f1 = float(row['f1_macro'].iloc[-1]) if len(row) else fallback
    ax.axhline(f1, color=color, ls=':', label=f'{label} (macro-F1 {f1:.2f})')

ax.set_xscale('log')
ax.set_xlabel('# training examples used (log scale)')
ax.set_ylabel('test score')
ax.set_title('Part 3 learning curve — BERT on Financial PhraseBank (allagree)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

cols = ['method', 'n_train_labeled', 'accuracy', 'f1_macro', 'f1_weighted']
pd.concat([curve[cols]] + [r[cols] for r in tech_rows] + ([llm[cols]] if len(llm) else []),
          ignore_index=True)

### 3c — technique comparison &nbsp;·&nbsp; 3d — methodology analysis

**3c (fold in Part 2 techniques):** the cell above overlays every Part 2 row logged in `results/results.csv` (`32-shot`, `augmented`, `llm-generated`, `optimal-combo`) on the data-scaling curve, each plotted at the number of training rows it used — i.e.‍ "how many real labels is each technique worth?".‍ Points appear automatically as 2d/2e get logged; re-run the cell after they finish.‍

_TODO (student-authored) — 3d methodology analysis._ Compare all methods (random / rule-based / 32-shot / augmentation / LLM-generated / full-data curve): where each helps, the data-efficiency story, limitations, and what you'd do with more compute.‍ Your words; commit with `--no-verify`.‍